# NB02 — Pangenome Classification Survey

Count defense/metabolism/homeostasis genes per representative genome (one per species, ~27K). Compute phylum-level prevalence.

In [1]:
import sys, os, re, warnings, requests, json, subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import fisher_exact, norm
from statsmodels.stats.multitest import fdrcorrection
warnings.filterwarnings("ignore")

spark = get_spark_session()
spark.clearProgressHandlers()  # suppress PlanMetrics objects that break nbconvert JSON serialization
# Patch JSON encoder to handle non-serializable Spark objects (PlanMetrics, etc.)
import json as _json
class _SparkSafeEncoder(_json.JSONEncoder):
    def default(self, obj):
        try:
            return super().default(obj)
        except TypeError:
            return str(obj)
_json.JSONEncoder = _SparkSafeEncoder
# Also patch the default encoder used by json.dumps
_json._default_encoder = _SparkSafeEncoder()

from pyspark.sql import functions as F

NOTEBOOK_DIR = Path().resolve()
PROJECT_DIR  = NOTEBOOK_DIR.parent
DATA_DIR     = PROJECT_DIR / "data"
FIG_DIR      = PROJECT_DIR / "figures"
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

def is_valid_parquet(p):
    if not p.exists() or p.stat().st_size < 512:
        return False
    try:
        import pyarrow.parquet as pq; pq.read_schema(str(p)); return True
    except Exception:
        return False

def wilson_ci(n, N, alpha=0.05):
    if N == 0: return np.nan, np.nan
    p = n / N; z = norm.ppf(1 - alpha/2)
    denom = 1 + z**2/N
    centre = (p + z**2/(2*N)) / denom
    half = z * np.sqrt(p*(1-p)/N + z**2/(4*N**2)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)

def odds_ratio_ci(a, b, c, d, alpha=0.05):
    # Woolf logit OR with 95% CI
    a,b,c,d = a+0.5, b+0.5, c+0.5, d+0.5
    log_or = np.log(a*d/(b*c))
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    z = norm.ppf(1 - alpha/2)
    return np.exp(log_or), np.exp(log_or - z*se), np.exp(log_or + z*se)

print("Setup complete.")

Setup complete.


In [2]:
spark.sql('''
    CREATE OR REPLACE TEMP VIEW one_per_species AS
    SELECT species, MIN(genome_id) AS rep_genome_id
    FROM   kbase.ke_pangenome.gtdb_taxonomy_r214v1
    WHERE  species IS NOT NULL
    GROUP BY species
''')
spark.sql('''
    CREATE OR REPLACE TEMP VIEW hq_one_per_species AS
    SELECT t.*
    FROM   kbase.ke_pangenome.gtdb_taxonomy_r214v1 t
    JOIN   one_per_species ops
           ON t.genome_id = ops.rep_genome_id AND t.species = ops.species
''')
n = spark.sql("SELECT COUNT(*) AS n FROM hq_one_per_species").collect()[0]["n"]
print(f"One-per-species set: {n:,} representative genomes")

One-per-species set: 27,690 representative genomes


## Section 1 — Load Vocabulary Map

In [3]:
DEFENSE_KOS   = ['K02585','K07633','K07634','K07635','K07636','K07637','K07638',
                  'K01551','K03455','K00537','K16551','K03756','K15523','K15524',
                  'K15525','K10783','K03078','K01533','K13579','K15724']
METABOLISM_KOS = ['K00114','K18155',
                   # pqqA-E (K11781-K11785) removed: pqqC absent throughout Thermoplasmatota,
                   # xoxF present in only 1/294; spurious annotation in non-methylotrophic archaea
                   'K02588','K02586','K02591','K22896','K22897','K22902','K16163',
                   'K01429','K01430','K01427','K00399','K14127','K05906']
HOMEOSTASIS_KOS = ['K06189','K03969','K03594','K04047','K09815','K09816','K09817',
                    'K07222','K15522','K18883','K23012']

DEFENSE_KW    = r'efflux|mercuric reductase|arsenate reductase|arsenical|chromate transporter|cadmium|copper-exporting|CusA|CzcA|CopA|ArsB|MerA|MerT|silA'
METABOLISM_KW = r'methanol dehydrogenase|nitrogenase|NiFe hydrogenase|urease|malonylpyruvate isomerase|xoxF|laccase|multicopper oxidase'
HOMEOSTASIS_KW = r'ferric uptake regulator|ferritin|bacterioferritin|dps protein|zinc ABC transporter|copper chaperone|manganese ABC'

def _ko_cond(kos):
    return ' OR '.join(f"eg.KEGG_ko LIKE '%{ko}%'" for ko in kos)

print(f"Seed KOs: {len(DEFENSE_KOS)} defense, {len(METABOLISM_KOS)} metabolism, {len(HOMEOSTASIS_KOS)} homeostasis")


Seed KOs: 20 defense, 15 metabolism, 11 homeostasis


## Section 2 — Count Genes per Genome

Join one-per-species representative genomes with gene annotations to compute metal gene counts.

In [4]:
genome_counts_path = DATA_DIR / "genome_metal_counts.parquet"
if is_valid_parquet(genome_counts_path):
    genome_counts_df = pd.read_parquet(genome_counts_path)
    print(f"[cache] genome_metal_counts: {len(genome_counts_df):,} genomes")
else:
    def_cond  = _ko_cond(DEFENSE_KOS)
    met_cond  = _ko_cond(METABOLISM_KOS)
    hom_cond  = _ko_cond(HOMEOSTASIS_KOS)

    genome_counts_df = spark.sql(f'''
        WITH ko_vocab AS (
            SELECT DISTINCT query_name AS gene_cluster_id,
                CASE
                    WHEN {def_cond} THEN 'defense'
                    WHEN {met_cond} THEN 'metabolism'
                    ELSE 'homeostasis'
                END AS category
            FROM kbase.ke_pangenome.eggnog_mapper_annotations eg
            WHERE eg.KEGG_ko IS NOT NULL AND eg.KEGG_ko != '-'
              AND ({def_cond} OR {met_cond} OR {hom_cond})
        ),
        kw_vocab AS (
            SELECT DISTINCT gene_cluster_id,
                CASE
                    WHEN product RLIKE '{DEFENSE_KW}'    THEN 'defense'
                    WHEN product RLIKE '{METABOLISM_KW}' THEN 'metabolism'
                    ELSE 'homeostasis'
                END AS category
            FROM kbase.ke_pangenome.bakta_annotations
            WHERE product IS NOT NULL
              AND (product RLIKE '{DEFENSE_KW}' OR product RLIKE '{METABOLISM_KW}' OR product RLIKE '{HOMEOSTASIS_KW}')
        ),
        vocab AS (
            SELECT gene_cluster_id, category FROM ko_vocab
            UNION ALL
            SELECT kw.gene_cluster_id, kw.category FROM kw_vocab kw
            WHERE NOT EXISTS (SELECT 1 FROM ko_vocab k WHERE k.gene_cluster_id = kw.gene_cluster_id)
        )
        SELECT
            ops.genome_id,
            ops.species,
            REGEXP_REPLACE(ops.phylum, '^p__', '') AS phylum,
            REGEXP_REPLACE(ops.class,  '^c__', '') AS class,
            REGEXP_REPLACE(ops.order,  '^o__', '') AS order_rank,
            REGEXP_REPLACE(ops.family, '^f__', '') AS family,
            REGEXP_REPLACE(ops.genus,  '^g__', '') AS genus,
            COALESCE(SUM(CASE WHEN v.category = 'defense'     THEN 1 ELSE 0 END), 0) AS n_defense,
            COALESCE(SUM(CASE WHEN v.category = 'metabolism'  THEN 1 ELSE 0 END), 0) AS n_metabolism,
            COALESCE(SUM(CASE WHEN v.category = 'homeostasis' THEN 1 ELSE 0 END), 0) AS n_homeostasis,
            COUNT(DISTINCT CASE WHEN v.category IS NOT NULL THEN gc.gene_cluster_id END) AS n_metal_total
        FROM hq_one_per_species ops
        JOIN kbase.ke_pangenome.genome g ON g.genome_id = ops.genome_id
        LEFT JOIN kbase.ke_pangenome.gene_cluster gc ON gc.gtdb_species_clade_id = g.gtdb_species_clade_id
        LEFT JOIN vocab v ON gc.gene_cluster_id = v.gene_cluster_id
        GROUP BY ops.genome_id, ops.species, ops.phylum, ops.class, ops.order, ops.family, ops.genus
    ''').toPandas()
    genome_counts_df.to_parquet(genome_counts_path, index=False)
    print(f"Genome counts: {len(genome_counts_df):,} genomes")

print(genome_counts_df[['n_defense','n_metabolism','n_homeostasis','n_metal_total']].describe())

Genome counts: 27,690 genomes
          n_defense  n_metabolism  n_homeostasis  n_metal_total
count  27690.000000  27690.000000   27690.000000   27690.000000
mean      23.092994      2.792452       5.010293      30.895739
std       34.820952      4.653109       4.805412      41.217205
min        0.000000      0.000000       0.000000       0.000000
25%        8.000000      0.000000       2.000000      11.000000
50%       16.000000      1.000000       4.000000      22.000000
75%       29.000000      4.000000       7.000000      39.000000
max     2949.000000    167.000000     230.000000    3346.000000


## Section 3 — Phylum-Level Prevalence

For major phyla, compute the prevalence (fraction with n_cat > 0) and Wilson 95% confidence intervals.

In [5]:
MAJOR_PHYLA = ['Actinomycetota', 'Pseudomonadota', 'Bacillota', 'Chloroflexota', 'Bacteroidota', 'Cyanobacteriota']

prevalence_rows = []
for phylum in MAJOR_PHYLA:
    phylum_data = genome_counts_df[genome_counts_df['phylum'] == phylum]
    n_total = len(phylum_data)
    
    if n_total > 0:
        for category in ['Defense', 'Metabolism', 'Homeostasis']:
            col = f"n_{category.lower()}"
            n_with = (phylum_data[col] > 0).sum()
            prev = n_with / n_total
            ci_low, ci_high = wilson_ci(n_with, n_total)
            
            prevalence_rows.append({
                'phylum': phylum,
                'category': category,
                'n_with_category': n_with,
                'n_total': n_total,
                'prevalence': prev,
                'ci_low': ci_low,
                'ci_high': ci_high
            })

prevalence_df = pd.DataFrame(prevalence_rows)
prevalence_df.to_csv(DATA_DIR / "phylum_prevalence.csv", index=False)

print("Phylum-level prevalence:")
pivot = prevalence_df.pivot_table(index='phylum', columns='category', values='prevalence', aggfunc='first')
print(pivot)

Phylum-level prevalence:
category          Defense  Homeostasis  Metabolism
phylum                                            
Actinomycetota   0.997793     0.925284    0.641551
Bacillota        0.979963     0.923113    0.355545
Bacteroidota     1.000000     0.985671    0.316616
Chloroflexota    1.000000     0.923414    0.540481
Cyanobacteriota  1.000000     0.782516    0.863539
Pseudomonadota   0.999464     0.985381    0.723176


## Section 4 — Grouped Bar Chart

Visualize prevalence by category and phylum with error bars.

In [6]:
fig, ax = plt.subplots(figsize=(12, 6))

phyla_list = MAJOR_PHYLA
categories = ['Defense', 'Metabolism', 'Homeostasis']
x = np.arange(len(phyla_list))
width = 0.25
colors = ['#e74c3c', '#3498db', '#2ecc71']

for i, cat in enumerate(categories):
    cat_data = prevalence_df[prevalence_df['category'] == cat].set_index('phylum').loc[phyla_list]
    prev_vals = cat_data['prevalence'].values
    errors = np.array([np.maximum(0, prev_vals - cat_data['ci_low'].values),
                       np.maximum(0, cat_data['ci_high'].values - prev_vals)])
    ax.bar(x + i*width, prev_vals, width, label=cat, color=colors[i],
           yerr=errors, capsize=5, error_kw={'ecolor': 'gray', 'elinewidth': 1.5})

ax.set_xlabel('Phylum')
ax.set_ylabel('Prevalence (fraction with gene)')
ax.set_title('Metal Gene Prevalence by Phylum')
ax.set_xticks(x + width)
ax.set_xticklabels(phyla_list, rotation=45, ha='right')
ax.legend()
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig(FIG_DIR / "nb02_phylum_prevalence.png", dpi=300, bbox_inches='tight')
plt.close()
print("Figure saved: nb02_phylum_prevalence.png")


Figure saved: nb02_phylum_prevalence.png
